# Song Popularity Classification

## CMOR 438 Final Project: Music Analytics with a From-Scratch Machine Learning Package

This notebook demonstrates binary classification of song popularity using the custom `LogisticRegression` implementation in `music_ml` and compares it against a scikit-learn baseline.

## 1) Problem Framing

Goal: predict whether a song belongs to a relatively high-popularity group based on audio features.

This formulation keeps the task interpretable and aligned with logistic regression assumptions.

## 2) Imports and Dataset Loading

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.metrics import (
    accuracy_score as sk_accuracy,
    precision_score as sk_precision,
    recall_score as sk_recall,
    f1_score as sk_f1,
)

# Make src package importable from notebooks/
sys.path.insert(0, str(Path("..").resolve() / "src"))

from music_ml.preprocessing import StandardScaler, train_test_split
from music_ml.supervised import LogisticRegression
from music_ml.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

data_path = Path("../data/processed/spotify_processed.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded dataset from: {data_path}")
    print(f"Shape: {df.shape}")
else:
    df = pd.DataFrame()
    print(
        "Dataset file not found. Place your processed CSV in data/processed/ "
        "and update data_path if needed."
    )

## 3) Select Numeric Audio Features

We select common Spotify-style numeric audio descriptors for modeling.

In [ ]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

if df.empty:
    print("No dataset loaded yet.")
    available_features = []
else:
    available_features = [col for col in audio_features if col in df.columns]
    print(f"Available feature count: {len(available_features)}")
    print("Features:", available_features)

## 4) Define a Binary Popularity Target

We create a binary target where `1` indicates popularity above a threshold.

By default, this notebook uses the median popularity as the threshold for a balanced setup.

In [ ]:
if df.empty:
    X = np.array([])
    y = np.array([])
    print("No dataset loaded yet.")
elif "popularity" not in df.columns:
    X = np.array([])
    y = np.array([])
    print("Column 'popularity' not found in dataset.")
elif len(available_features) == 0:
    X = np.array([])
    y = np.array([])
    print("No required audio features found in dataset.")
else:
    modeling_df = df[available_features + ["popularity"]].dropna().copy()

    threshold = modeling_df["popularity"].median()
    modeling_df["popular_binary"] = (modeling_df["popularity"] >= threshold).astype(int)

    X = modeling_df[available_features].to_numpy(dtype=float)
    y = modeling_df["popular_binary"].to_numpy(dtype=int)

    print(f"Modeling samples: {len(modeling_df)}")
    print(f"Popularity threshold (median): {threshold:.3f}")
    print("Class balance:")
    print(pd.Series(y).value_counts(normalize=True).rename("proportion"))

## 5) Train/Test Split

In [ ]:
if X.size == 0:
    print("Training data is not ready.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        shuffle=True,
        random_state=42,
    )
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape:  {X_test.shape}")

## 6) Standardization with Custom StandardScaler

In [ ]:
if X.size == 0:
    print("Training data is not ready.")
else:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("Scaling complete.")
    print("Train means (approx. 0):", np.round(X_train_scaled.mean(axis=0), 3))

## 7) Train Custom LogisticRegression Model

In [ ]:
if X.size == 0:
    print("Training data is not ready.")
else:
    custom_model = LogisticRegression(learning_rate=0.05, n_iters=3000)
    custom_model.fit(X_train_scaled, y_train)

    y_pred_custom = custom_model.predict(X_test_scaled)
    y_prob_custom = custom_model.predict_proba(X_test_scaled)

    print("Custom model trained.")
    print(f"Iterations run: {custom_model.n_iters}")
    print(f"Final training loss: {custom_model.loss_history_[-1]:.4f}")

## 8) Evaluate with Custom Metrics

We evaluate classification performance using metrics from `music_ml.metrics`.

In [ ]:
if X.size == 0:
    print("Training data is not ready.")
else:
    custom_results = {
        "accuracy": accuracy_score(y_test, y_pred_custom),
        "precision": precision_score(y_test, y_pred_custom),
        "recall": recall_score(y_test, y_pred_custom),
        "f1": f1_score(y_test, y_pred_custom),
    }

    print("Custom LogisticRegression Metrics")
    for metric_name, metric_value in custom_results.items():
        print(f"{metric_name:>10}: {metric_value:.4f}")

    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred_custom))

## 9) Compare with scikit-learn LogisticRegression

This baseline provides a reference point for validating the custom implementation.

In [ ]:
if X.size == 0:
    print("Training data is not ready.")
else:
    sk_model = SklearnLogisticRegression(max_iter=2000, random_state=42)
    sk_model.fit(X_train_scaled, y_train)
    y_pred_sk = sk_model.predict(X_test_scaled)

    comparison = pd.DataFrame(
        {
            "custom_music_ml": [
                custom_results["accuracy"],
                custom_results["precision"],
                custom_results["recall"],
                custom_results["f1"],
            ],
            "sklearn_baseline": [
                sk_accuracy(y_test, y_pred_sk),
                sk_precision(y_test, y_pred_sk),
                sk_recall(y_test, y_pred_sk),
                sk_f1(y_test, y_pred_sk),
            ],
        },
        index=["accuracy", "precision", "recall", "f1"],
    )

    display(comparison)

## 10) Coefficient Interpretation and Limitations

The learned coefficients indicate how each standardized feature contributes to the log-odds of being in the high-popularity class.

Use this section to report the strongest positive and negative contributors and connect them to musical interpretation.

In [ ]:
if X.size == 0:
    print("Training data is not ready.")
else:
    coef_table = pd.DataFrame(
        {
            "feature": available_features,
            "coefficient": custom_model.weights_,
            "abs_coefficient": np.abs(custom_model.weights_),
        }
    ).sort_values("abs_coefficient", ascending=False)

    display(coef_table)

    print("Suggested discussion points:")
    print("- Which features have the largest positive coefficients?")
    print("- Which features have the largest negative coefficients?")
    print("- How sensitive are results to target threshold choice?")
    print("- What nonlinear effects or interactions might logistic regression miss?")